# Climate State Classifier - Inference and Evaluation Demo

This notebook loads the trained demo model and runs inference on the demo data.
It generates prediction tables, a CSV file, and explanation plots, then displays them inline.


In [ ]:
from pathlib import Path
import os
import sys

repo_root = Path.cwd().resolve()
if repo_root.name == "demo":
    repo_root = repo_root.parent
os.chdir(repo_root)
sys.path.insert(0, str(repo_root))

print("Repo root:", repo_root)
print("Working dir:", Path.cwd())


## Check For A Trained Model


In [ ]:
model_path = repo_root / "demo" / "snapshots" / "100.pth"
print("Model exists:", model_path.exists(), model_path)
if not model_path.exists():
    print("Model not found. Run the training notebook first.")


## Evaluation Configuration


In [ ]:
print((repo_root / "demo" / "demo_evaluate.txt").read_text())


## Run Inference And Collect Outputs


In [ ]:
from climatestateclassifier import config as cfg
from climatestateclassifier.evaluate import create_prediction
from climatestateclassifier.utils import plot_utils

cfg.set_evaluate_args("demo/demo_evaluate.txt")

Path(cfg.eval_dir).mkdir(parents=True, exist_ok=True)
(Path(cfg.eval_dir) / "tables").mkdir(parents=True, exist_ok=True)
(Path(cfg.eval_dir) / "explanations").mkdir(parents=True, exist_ok=True)

model_file = str(Path(cfg.model_dir) / f"{cfg.model_names[0]}.pth")

inputs, outputs, labels, categories, sample_names, dims, explanations = create_prediction(
    model_file, cfg.val_samples
)


## Create A CSV With Per-Sample Predictions


In [ ]:
import pandas as pd
import torch

probs = torch.softmax(outputs, dim=1)
pred_idx = probs.argmax(1)
gt_idx = labels.argmax(1)

rows = []
for i in range(len(sample_names)):
    gt_i = int(gt_idx[i])
    pred_i = int(pred_idx[i])
    row = {
        "sample": sample_names[i],
        "category": categories[i],
        "ground_truth": cfg.label_names[gt_i],
        "prediction": cfg.label_names[pred_i],
    }
    for j, name in enumerate(cfg.label_names):
        row[f"prob_{name}"] = float(probs[i, j])
    rows.append(row)

pred_df = pd.DataFrame(rows)
csv_path = Path(cfg.eval_dir) / "predictions.csv"
pred_df.to_csv(csv_path, index=False)

print("Wrote:", csv_path)
pred_df


## Generate Tables And Explanations Using Built-In Plotting


In [ ]:
eval_name = cfg.eval_names[0]

plot_utils.plot_prediction_overview(outputs, labels, eval_name=eval_name)
plot_utils.plot_class_predictions(outputs, labels, eval_name=eval_name)
plot_utils.plot_predictions_by_category(outputs, labels, categories, eval_name=eval_name)
plot_utils.plot_single_predictions(outputs, labels, categories, sample_names, eval_name=eval_name)

plot_utils.plot_explanations(
    inputs, dims, labels, outputs, sample_names, categories, explanations, eval_name=eval_name
)

print("Tables:", sorted((Path(cfg.eval_dir) / "tables").glob("*.pdf")))
print("Explanations:", sorted((Path(cfg.eval_dir) / "explanations").glob("*.jpg")))


## Display Tables Inline


In [ ]:
import matplotlib.pyplot as plt

def show_table(df, title):
    fig, ax = plt.subplots(figsize=(max(6, len(df.columns) * 2), max(1, len(df) * 0.5 + 1)))
    ax.axis("off")
    table = ax.table(cellText=df.values, colLabels=df.columns, loc="center")
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1, 1.5)
    ax.set_title(title)
    plt.show()

label_names = list(cfg.label_names)
n_labels = len(label_names)

pred_idx_np = pred_idx.cpu().numpy()
gt_idx_np = gt_idx.cpu().numpy()

total_gt = [(gt_idx_np == i).sum() for i in range(n_labels)]
total_pred = [(pred_idx_np == i).sum() for i in range(n_labels)]
correct = [((gt_idx_np == i) & (pred_idx_np == i)).sum() for i in range(n_labels)]
false = [((gt_idx_np != i) & (pred_idx_np == i)).sum() for i in range(n_labels)]

overview_df = pd.DataFrame({
    "Label": label_names + ["Total"],
    "Total Ground Truth": total_gt + [sum(total_gt)],
    "Total Predicted": total_pred + [sum(total_pred)],
    "Correct": correct + [sum(correct)],
    "False": false + [sum(false)],
})

show_table(overview_df, "Overview")
overview_df


## Confusion Matrix


In [ ]:
cm = pd.DataFrame(0, index=label_names, columns=label_names)
for gt, pred in zip(gt_idx_np, pred_idx_np):
    cm.iloc[gt, pred] += 1

show_table(cm, "Confusion Matrix (Counts)")
cm


## Predictions By Category


In [ ]:
rows = []
for cat in cfg.val_categories:
    for gt_label_idx, gt_label in enumerate(label_names):
        idx = [i for i in range(len(sample_names)) if categories[i] == cat and gt_idx_np[i] == gt_label_idx]
        total = len(idx)
        row = {"Category": cat, "Ground Truth": gt_label}
        for pred_label_idx, pred_label in enumerate(label_names):
            count = sum(1 for i in idx if pred_idx_np[i] == pred_label_idx)
            row[pred_label] = 0 if total == 0 else round(100 * count / total, 2)
        rows.append(row)

category_df = pd.DataFrame(rows)
show_table(category_df, "Predictions By Category (Percent)")
category_df


## Per-Sample Predictions


In [ ]:
show_table(pred_df[["sample", "category", "ground_truth", "prediction"]], "Per-Sample Predictions")
pred_df


## Explanation Plots


In [ ]:
from IPython.display import Image, display

for img_path in sorted((Path(cfg.eval_dir) / "explanations").glob("*.jpg")):
    display(Image(filename=str(img_path)))
